# 05 · El compromiso utilidad-privacidad y el semáforo

Síntesis del estudio piloto: ningún generador es a la vez útil para inferencia clínica y seguro
frente a los ataques en todos los estudios.

In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from synapse import datasets, generators, inference as inf, utility as ut, privacy as pv, compliance, viz
from synapse import experiment as ex
viz.style(); pd.set_option("display.precision", 3); pd.set_option("display.width", 140)
TABLES = ROOT / "results" / "tables"
metrics = pd.read_csv(TABLES / "study_metrics.csv") if (TABLES / "study_metrics.csv").exists() else None

In [2]:
print(len(metrics), "réplicas ·", metrics["dataset"].nunique(), "estudios ·", metrics["label"].nunique(), "generadores")
ex.summarize(metrics)

320 réplicas · 4 estudios · 8 generadores


,dataset,label,comb_ci_overlap_mean,ci_overlap_mean,decisions_changed,km_mean_abs_diff,pmse_ratio,cindex_tstr,cindex_gap,mia_dist_auc,mia_dens_auc,dcr_share_excess,exact_match_rate,singling_out_risk,linkability_risk,inference_risk,verde_%
0,flchain,bootstrap,0.834 ± 0.023,0.762 ± 0.027,0.2 ± 0.133,0.008 ± 0.002,0.933 ± 0.09,0.799 ± 0.004,0.001 ± 0.001,0.813 ± 0.004,0.6 ± 0.004,0.5 ± 0.0,1.0 ± 0.0,0.752 ± 0.008,0.628 ± 0.02,0.656 ± 0.02,0
1,flchain,cart,0.606 ± 0.083,0.619 ± 0.063,0.8 ± 0.249,0.013 ± 0.002,1.25 ± 0.143,0.8 ± 0.003,0.0 ± 0.001,0.564 ± 0.005,0.533 ± 0.005,0.12 ± 0.004,0.007 ± 0.001,0.757 ± 0.006,0.19 ± 0.036,0.21 ± 0.017,0
2,flchain,copula,0.572 ± 0.018,0.512 ± 0.024,0.6 ± 0.221,0.041 ± 0.003,0.961 ± 0.115,0.797 ± 0.004,0.003 ± 0.002,0.503 ± 0.004,0.508 ± 0.005,0.005 ± 0.007,0.0 ± 0.0,0.048 ± 0.007,0.013 ± 0.007,0.055 ± 0.018,10
3,flchain,dpcopula(ε=0.5),0.294 ± 0.03,0.258 ± 0.02,1.2 ± 0.359,0.044 ± 0.009,192.116 ± 8.059,0.596 ± 0.074,0.204 ± 0.071,0.496 ± 0.005,0.5 ± 0.006,0.01 ± 0.01,0.0 ± 0.0,0.004 ± 0.002,0.011 ± 0.004,0.033 ± 0.011,0
4,flchain,dpcopula(ε=1),0.298 ± 0.028,0.27 ± 0.027,1.0 ± 0.333,0.039 ± 0.006,143.163 ± 7.008,0.608 ± 0.063,0.192 ± 0.061,0.498 ± 0.003,0.506 ± 0.004,0.007 ± 0.008,0.0 ± 0.0,0.004 ± 0.002,0.014 ± 0.007,0.057 ± 0.023,0
5,flchain,dpcopula(ε=5),0.424 ± 0.027,0.398 ± 0.026,0.7 ± 0.26,0.037 ± 0.004,34.229 ± 2.691,0.783 ± 0.004,0.016 ± 0.003,0.499 ± 0.003,0.509 ± 0.005,0.005 ± 0.007,0.0 ± 0.0,0.005 ± 0.003,0.017 ± 0.01,0.045 ± 0.016,0
6,flchain,dpcopula(ε=10),0.462 ± 0.022,0.421 ± 0.028,0.8 ± 0.291,0.035 ± 0.003,16.1 ± 1.291,0.787 ± 0.005,0.012 ± 0.003,0.504 ± 0.003,0.51 ± 0.005,0.014 ± 0.005,0.0 ± 0.0,0.007 ± 0.003,0.015 ± 0.008,0.05 ± 0.019,0
7,flchain,marginal,0.492 ± 0.024,0.486 ± 0.026,2.1 ± 0.314,0.089 ± 0.003,1.383 ± 0.099,0.467 ± 0.068,0.332 ± 0.069,0.503 ± 0.003,0.506 ± 0.006,0.008 ± 0.007,0.0 ± 0.0,0.049 ± 0.005,0.019 ± 0.007,0.038 ± 0.013,0
8,gbsg2,bootstrap,0.898 ± 0.008,0.764 ± 0.023,0.6 ± 0.221,0.017 ± 0.003,1.104 ± 0.099,0.664 ± 0.007,0.009 ± 0.004,0.812 ± 0.005,0.55 ± 0.01,0.499 ± 0.0,1.0 ± 0.0,0.799 ± 0.01,0.6 ± 0.014,0.632 ± 0.015,0
9,gbsg2,cart,0.808 ± 0.014,0.669 ± 0.026,1.5 ± 0.307,0.024 ± 0.004,1.006 ± 0.108,0.648 ± 0.008,0.024 ± 0.006,0.581 ± 0.005,0.53 ± 0.01,0.131 ± 0.008,0.001 ± 0.0,0.798 ± 0.01,0.167 ± 0.029,0.159 ± 0.033,0


## Frente utilidad-privacidad por estudio

In [3]:
for name in metrics["dataset"].unique():
    viz.pareto(metrics, name)

## Coste inferencial de la privacidad diferencial

In [4]:
viz.epsilon_sweep(metrics);

## Semáforo de liberación

In [5]:
viz.verdict_map(metrics);

## Hipótesis del piloto (exploratorias; las confirmatorias se preregistran)

In [6]:
m = metrics.assign(n_small=metrics["n_train"] < 1000)
h1 = m[m["generator"].isin(["cart", "copula"])].groupby(["dataset", "generator"])["comb_ci_overlap_mean"].mean().unstack()
h2 = m[m["dp"] & (m["epsilon"] <= 1)].groupby("dataset")["comb_ci_overlap_mean"].mean()
h3 = m[m["generator"].isin(["bootstrap", "copula", "cart", "marginal"])].groupby("generator")["singling_out_risk"].mean()
display(h1.round(3), h2.round(3).to_frame("solapamiento con ε ≤ 1"), h3.round(3).to_frame("individualización media"))

generator,cart,copula
dataset,,
flchain,0.606,0.572
gbsg2,0.808,0.754
veterans,0.764,0.743
whas500,0.736,0.693


,solapamiento con ε ≤ 1
dataset,
flchain,0.296
gbsg2,0.298
veterans,0.257
whas500,0.237


,individualización media
generator,
bootstrap,0.727
cart,0.653
copula,0.094
marginal,0.093


**Lectura (piloto).**

* **H1** (CART conserva mejor los HR que los métodos profundos con n < 1.000): pendiente de los
  generadores del extra `[deep]`; frente a la cópula, CART conserva mejor los efectos de Cox.
* **H2** (privacidad diferencial con ε ≤ 1 rompe la validez inferencial): consistente. El
  solapamiento cae por debajo de 0,3 en todos los estudios.
* **H3** (bootstrap y CART fallan la individualización): consistente, con riesgo cercano a 0,75.

> 📝 **Para Maricel:** redactar estas lecturas en la sección de Resultados del paper
> (`paper/paper.qmd`) y fijar en el preregistro las semillas del estudio confirmatorio
> (por ejemplo, 100–199), distintas de las del piloto (0–9).